# Week 3

### Goals
- Load a real film catalog and build a classical recommendation baseline.
- Understand what information is actually available in each file

### Tasks
- Download MovieLens latest-small from GroupLens and inspect movies.csv, ratings.csv, tags.csv and links.csv.
- Load CSV files with pandas; join rating summaries to movie metadata by movieId.
- Clean titles and split the pipe-separated genre field.
- Create a manageable catalog subset and a function to filter by genre and year.
- Rank candidates by a simple transparent score, while accounting for few ratings.
- Keep unknown fields such as runtime and plot marked as unknown rather than inventing them.

### Expected Output
- 03_movie_data.ipynb and a documented local catalog.
- A rule-based recommend_movies() baseline with example results.

---
### Data preparation

#### Load and inspect datasets

In [19]:
import pandas as pd

movies_df = pd.read_csv('../data/ml-latest-small/movies.csv', sep=',')
ratings_df = pd.read_csv('../data/ml-latest-small/ratings.csv', sep=',')

display(movies_df.head(10))
display(ratings_df.head(10))


display(movies_df.info())
display(ratings_df.info())

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 626.1 KB


None

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


None

In [20]:
# Calculate average rating and number of ratings per movie
rating_summaries = ratings_df.groupby('movieId')['rating'].agg(['mean', 'count']).reset_index()
display(rating_summaries.head(10))
display(rating_summaries.info())

,movieId,mean,count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49
5,6,3.946078,102
6,7,3.185185,54
7,8,2.875000,8
8,9,3.125000,16
9,10,3.496212,132


<class 'pandas.DataFrame'>
RangeIndex: 9724 entries, 0 to 9723
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9724 non-null   int64  
 1   mean     9724 non-null   float64
 2   count    9724 non-null   int64  
dtypes: float64(1), int64(2)
memory usage: 228.0 KB


None

#### Merge datasets

In [21]:
# Merge the two dataframes on the 'movieId' column
merged_df = pd.merge(movies_df, rating_summaries, on='movieId')
display(merged_df.head(10))
print(merged_df.info())

,movieId,title,genres,mean,count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49
5,6,Heat (1995),Action|Crime|Thriller,3.946078,102
6,7,Sabrina (1995),Comedy|Romance,3.185185,54
7,8,Tom and Huck (1995),Adventure|Children,2.875000,8
8,9,Sudden Death (1995),Action,3.125000,16
9,10,GoldenEye (1995),Action|Adventure|Thriller,3.496212,132


<class 'pandas.DataFrame'>
RangeIndex: 9724 entries, 0 to 9723
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9724 non-null   int64  
 1   title    9724 non-null   str    
 2   genres   9724 non-null   str    
 3   mean     9724 non-null   float64
 4   count    9724 non-null   int64  
dtypes: float64(1), int64(2), str(2)
memory usage: 779.3 KB
None


#### Clean merged dataframe
- Title is often unclean, consisting of unecessary words or characters and the year of the movie
- Adjust genres column to hold a list of geners instead of the geners being separated by the pipe operator

In [22]:
# Extract the 4 digits inside the last set of parentheses into a new 'year' column and convert it to integer 
merged_df['year'] = merged_df['title'].str.extract(r'\((\d{4})\)').astype(float).astype('Int64')

# Remove the year and any trailing spaces from the 'title' column
merged_df['title'] = merged_df['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True)

# Remove "a.k.a." and anything inside its parantheses from the 'title' column
merged_df['title'] = merged_df['title'].str.replace(r'\s*\(a\.k\.a\.[^)]*\)', '', regex=True)

# Remove other trailing alternative language titles in parentheses.
merged_df['title'] = merged_df['title'].str.replace(r'\s*\([^)]*\)$', '', regex=True)

# 3. Move prefixes like ", The", ", A", or ", An" from the end of the title to the beginning.
merged_df['title'] = merged_df['title'].str.replace(r'^(.*),\s*(The|A|An)\s*$', r'\2 \1', regex=True)

# Strip any lingering whitespace just to be perfectly clean
merged_df['title'] = merged_df['title'].str.strip()

# Split the pipe-separated genre field into a list of strings
merged_df['genres'] = merged_df['genres'].str.split('|')

display(merged_df.head())
display(merged_df.info())

,movieId,title,genres,mean,count,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",3.920930,215,1995
1,2,Jumanji,"[Adventure, Children, Fantasy]",3.431818,110,1995
2,3,Grumpier Old Men,"[Comedy, Romance]",3.259615,52,1995
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",2.357143,7,1995
4,5,Father of the Bride Part II,[Comedy],3.071429,49,1995


<class 'pandas.DataFrame'>
RangeIndex: 9724 entries, 0 to 9723
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9724 non-null   int64  
 1   title    9724 non-null   str    
 2   genres   9724 non-null   object 
 3   mean     9724 non-null   float64
 4   count    9724 non-null   int64  
 5   year     9711 non-null   Int64  
dtypes: Int64(1), float64(1), int64(2), object(1), str(1)
memory usage: 620.6+ KB


None

In [23]:
# Check the rows with missing values in the 'year' column
missing_years = merged_df[merged_df['year'].isnull()]
display(missing_years)

,movieId,title,genres,mean,count,year
6042,40697,Babylon 5,[Sci-Fi],2.25,2,<NA>
9013,140956,Ready Player One,"[Action, Sci-Fi, Thriller]",3.50,4,<NA>
9073,143410,Hyena Road,[(no genres listed)],2.00,1,<NA>
9120,147250,The Adventures of Sherlock Holmes and Doctor W...,[(no genres listed)],5.00,1,<NA>
9161,149334,Nocturnal Animals,"[Drama, Thriller]",3.00,1,<NA>
9241,156605,Paterson,[(no genres listed)],4.50,1,<NA>
9349,162414,Moonlight,[Drama],5.00,1,<NA>
9430,167570,The OA,[(no genres listed)],4.00,1,<NA>
9496,171495,Cosmos,[(no genres listed)],4.50,2,<NA>
9497,171631,Maria Bamford: Old Baby,[(no genres listed)],1.00,1,<NA>


It's only very few movies with missing years. I will keep to the original CSV and leave them NA for now. Alternatively it would also be possible to fill those years manually by looking up the release years.

Checking these NAs reveald and additional small issue in the genres column, not found before. There are some movies with a "(no genres listed)" argument as genres. The creators of this dataset probably used this to flag missing data instead of using NA. Keeping it as it is could lead the future filtering logic to accidentally treating "(no genres listed)" as an actual genre category. To prevent this from happening,  I additionally convert all ['(no genres listed)'] entries in genres to empty lists.

In [24]:
# Convert ['(no genres listed)'] to an empty list
merged_df['genres'] = merged_df['genres'].apply(lambda x: [] if x == ['(no genres listed)'] else x)

# Check same rows with missing year values again
missing_years = merged_df[merged_df['year'].isnull()]
display(missing_years)

,movieId,title,genres,mean,count,year
6042,40697,Babylon 5,[Sci-Fi],2.25,2,<NA>
9013,140956,Ready Player One,"[Action, Sci-Fi, Thriller]",3.50,4,<NA>
9073,143410,Hyena Road,[],2.00,1,<NA>
9120,147250,The Adventures of Sherlock Holmes and Doctor W...,[],5.00,1,<NA>
9161,149334,Nocturnal Animals,"[Drama, Thriller]",3.00,1,<NA>
9241,156605,Paterson,[],4.50,1,<NA>
9349,162414,Moonlight,[Drama],5.00,1,<NA>
9430,167570,The OA,[],4.00,1,<NA>
9496,171495,Cosmos,[],4.50,2,<NA>
9497,171631,Maria Bamford: Old Baby,[],1.00,1,<NA>


#### Save cleaned dataset to new CSV

In [25]:
# Save the cleaned and merged dataframe to a new CSV file
merged_df.to_csv('../data/cleaned_movie_data.csv', index=False)

--- 
### Movie recommendation function

In [26]:
# First, check stats of number of ratings per movie
display(merged_df['count'].describe())

count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
Name: count, dtype: float64

In [27]:
def recommend_movies(catalog_df, target_genre=None, target_year=None, min_ratings=3, top_n=5):
    # Create a copy so the original dataframe is not modified
    candidates = catalog_df.copy()
    
    # Account for few ratings by applying the minimum threshold
    candidates = candidates[candidates['count'] >= min_ratings]
    
    # Filter by year
    if target_year is not None:
        candidates = candidates[candidates['year'] == target_year]
        
    # Filter by genre (checking if the target_genre is in the list of genres)
    if target_genre is not None:
        candidates = candidates[candidates['genres'].apply(lambda x: target_genre in x)]

    # If no candidates remain after filtering, return an empty list
    if candidates.empty:
        return []
        
    # Rank candidates by simple transparent score (mean rating)
    candidates = candidates.sort_values(by='mean', ascending=False)
    
    # Take the top N rows BEFORE converting to a list of dictionaries
    top_candidates = candidates.head(top_n).to_dict('records')
    
    # Keep unknown fields marked as unknown
    for row in top_candidates:
        row['runtime'] = 'Unknown'
        row['plot'] = 'Unknown'

    return top_candidates

In [28]:
# test the function with some example parameters
recommend_movies(merged_df, target_genre='Action', target_year=2000, min_ratings=5, top_n=3)

[{'movieId': 3275,
  'title': 'The Boondock Saints',
  'genres': ['Action', 'Crime', 'Drama', 'Thriller'],
  'mean': 4.22093023255814,
  'count': 43,
  'year': 2000,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 3578,
  'title': 'Gladiator',
  'genres': ['Action', 'Adventure', 'Drama'],
  'mean': 3.9382352941176473,
  'count': 170,
  'year': 2000,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 7022,
  'title': 'Battle Royale',
  'genres': ['Action', 'Drama', 'Horror', 'Thriller'],
  'mean': 3.933333333333333,
  'count': 30,
  'year': 2000,
  'runtime': 'Unknown',
  'plot': 'Unknown'}]

In [29]:
# test the function with some example parameters
recommend_movies(merged_df, target_genre='Fantasy', target_year=None, min_ratings=8, top_n=5)

[{'movieId': 62336,
  'title': 'FLCL',
  'genres': ['Animation', 'Comedy', 'Fantasy', 'Sci-Fi'],
  'mean': 4.277777777777778,
  'count': 9,
  'year': 2000,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 1197,
  'title': 'The Princess Bride',
  'genres': ['Action', 'Adventure', 'Comedy', 'Fantasy', 'Romance'],
  'mean': 4.232394366197183,
  'count': 142,
  'year': 1987,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 1199,
  'title': 'Brazil',
  'genres': ['Fantasy', 'Sci-Fi'],
  'mean': 4.177966101694915,
  'count': 59,
  'year': 1985,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 1136,
  'title': 'Monty Python and the Holy Grail',
  'genres': ['Adventure', 'Comedy', 'Fantasy'],
  'mean': 4.161764705882353,
  'count': 136,
  'year': 1975,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 122926,
  'title': 'Untitled Spider-Man Reboot',
  'genres': ['Action', 'Adventure', 'Fantasy'],
  'mean': 4.15625,
  'count': 16,
  'year': 2017,
  'runtim

In [30]:
# test the function with some example parameters
recommend_movies(merged_df, target_genre=['Comedy', 'Romance'], target_year=None, min_ratings=3, top_n=5)

[]

In [31]:
# test the function with some example parameters
recommend_movies(merged_df, target_genre='Comedy', target_year=2020, min_ratings=3, top_n=5)

[]

#### Observations
This very simple baseline function works as long as the inputs are also simple and inside the functions scope. For example searching for one specific genre or year (or combination) works perfectly fine and the `min_ratings` and `top_n` parameters are just filters for the final returns. But with this baseline it is not possible to for example search for multiple categories (e.g. an action-comedy movie), also it is only possible to search for a specific year or none at all. Finding a movie in between certain years or newer/older than year x is not possible. Such additions need to be made for a final, reworked recommendation system. Also when a no result is found the return is just an empty list. How the agent then handles an empty return needs to be cleary specified. 

---

**Dataset Citation**  
F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. https://doi.org/10.1145/2827872